# Day 2 - House Price Prediction

Predicting house prices using regression. Three models compared: Linear Regression, Random Forest, and Gradient Boosting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline

## 1. Load Data

In [ ]:
df = pd.read_csv('dataset/housing.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.describe().round(2)

In [ ]:
print('Missing values:')
print(df.isnull().sum())

## 2. EDA

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df['price'], bins=40, color='#2196F3', alpha=0.8, edgecolor='white')
plt.title('Price Distribution')
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['sqft_living'], df['price'], alpha=0.3, s=10, color='#FF5722')
plt.xlabel('Living Area (sqft)')
plt.ylabel('Price ($)')
plt.title('Sqft vs Price')
plt.show()

# strong linear relationship as expected

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()

## 3. Train / Test Split

In [ ]:
feature_cols = ['bedrooms', 'bathrooms', 'sqft_living', 'age_years',
                'garage_spaces', 'floors', 'location_score', 'distance_to_city_km']

X = df[feature_cols]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Train:', X_train.shape)
print('Test: ', X_test.shape)

## 4. Model Comparison

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    print(f'{name}')
    print(f'  MAE:  ${mae:,.0f}  |  RMSE: ${rmse:,.0f}  |  R2: {r2:.4f}\n')

## 5. Best Model Analysis

In [ ]:
best = models['Gradient Boosting']
preds = best.predict(X_test_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(y_test, preds, alpha=0.3, color='#FF5722', s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', linewidth=1)
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title('Actual vs Predicted')
plt.show()

In [ ]:
importances = best.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

plt.figure(figsize=(8, 5))
plt.bar([feature_cols[i] for i in sorted_idx], importances[sorted_idx], color='#4CAF50')
plt.title('Feature Importance')
plt.ylabel('Importance')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

# sqft_living dominates - makes sense